# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset entities such as record sets, fields, and columns are referenced by their `@id` values.

### Dataset Source
The dataset is defined by a Croissant schema available at the following URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading

We load the dataset metadata and records using `mlcroissant`. The metadata object gives us access to the dataset's description, and the record sets, fields, and columns are all referenced by their `@id`s for reproducibility.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# NOTE: .metadata is an object, not a dict or list
print(f"Dataset loaded: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview

We will enumerate the available Record Sets, their Fields (with their `@id`s), and preview example records. All references to dataset entities are using their `@id` values.


In [ ]:
print("Record sets available in this dataset:")
for record_set in dataset.metadata.record_sets:
    print(f"  RecordSet @id: {record_set.id}  |  name: {record_set.name}")

print("\nListing fields and their @id for each record set:\n")

for record_set in dataset.metadata.record_sets:
    print(f"Record Set @id: {record_set.id} (name: {record_set.name})")
    if hasattr(record_set, 'fields') and record_set.fields:
        for field in record_set.fields:
            print(f"   Field @id: {field.id:42}  name: {field.name}   datatype: {field.data_type}")
    if hasattr(record_set, 'columns') and record_set.columns:
        print("   Columns:")
        for col in record_set.columns:
            print(f"     Column @id: {col.id:38}  name: {col.name}   datatype: {col.data_type}")
    print("\nExample records from this record set:")
    # Print a preview of example records
    try:
        records = list(dataset.records(record_set=record_set.id))
        for example in records[:3]:
            print(json.dumps(example, indent=2)[:800])
            print("---")
    except Exception as e:
        print(f"Could not load records for {record_set.id}: {str(e)}")
    print("\n" + ("=" * 60) + "\n")

## 3. Data Extraction

We load data from each record set into a separate pandas DataFrame. All entity references—including record set IDs and column/field names—are managed dynamically based on the loaded metadata structure, and always referenced by their `@id` for reproducibility.


In [ ]:
# List all record set @id's
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for rs_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from RecordSet @id: {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2), "\n---\n")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {str(e)}\n")

## 4. Exploratory Data Analysis (EDA)

Let us select one record set for analysis, and filter, normalize, and group by relevant fields. All references are by `@id` values.


In [ ]:
# Pick the main (largest) record set (usually the main data table)
main_rs_id = None
max_rows = 0
for rs_id, df in dataframes.items():
    if len(df) > max_rows:
        main_rs_id = rs_id
        max_rows = len(df)

main_df = dataframes[main_rs_id]

print(f"Selected RecordSet for EDA: {main_rs_id} with {len(main_df)} rows.")
print(f"Fields: {main_df.columns.tolist()}")

# Automatically try to pick a numeric field using dtype
numeric_candidates = main_df.select_dtypes(include=['number']).columns.tolist()
if not numeric_candidates:
    # Try to parse numbers from string columns
    for col in main_df.columns:
        try:
            main_df[col] = pd.to_numeric(main_df[col], errors='ignore')
        except:
            continue
    numeric_candidates = main_df.select_dtypes(include=['number']).columns.tolist()

if numeric_candidates:
    numeric_field = numeric_candidates[0]  # Pick the first numeric
else:
    raise RuntimeError("Could not find a numeric field for EDA.")

print(f"Using numeric field @id (column name): {numeric_field}")

# Example: set a sensible threshold (using mean, median, or 0 if dtype appropriate)
if main_df[numeric_field].dtype.kind in 'ifc':
    threshold = main_df[numeric_field].median()  # Median as an example threshold
else:
    threshold = 0

filtered_df = main_df[main_df[numeric_field] > threshold].copy()

print(f"Filtered to {len(filtered_df)} records with {numeric_field} > {threshold}")

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
)
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to choose a categorical field to group by
non_numeric_fields = [col for col in main_df.columns if col not in numeric_candidates and main_df[col].nunique() < 20]
if non_numeric_fields:
    group_field = non_numeric_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("No categorical field found to group by.")

## 5. Visualization

We visualize the distribution of the selected numeric field and, if a grouping categorical field exists, compare means across groups.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram for the chosen numeric field
plt.figure(figsize=(7,4))
sns.histplot(main_df[numeric_field], bins=30, kde=True)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field} in RecordSet @{main_rs_id}")
plt.show()

# If we found a group field, visualize means
if 'group_field' in locals():
    plt.figure(figsize=(8,5))
    means = grouped_df.sort_values(numeric_field, ascending=False)
    sns.barplot(y=means.index, x=means[numeric_field], palette="viridis")
    plt.xlabel(f"Mean {numeric_field}")
    plt.ylabel(group_field)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing all entities by their `@id`. We performed data preview, dynamic extraction, filtering, normalization, grouping, and visualization of the dataset's main numeric fields. For deeper analysis, consult the Croissant schema or refer to specific `@id`s listed in the overview cells above.